# Feature Exploration: Delay Rate Acceleration (Multi-Route)

This notebook tests the impact of the delay rate acceleration feature (second derivative) on the multi-route model. THhe baseline model used here is like in 7d but with added the number extreme weather days as a feature.

**New Feature to Test:**
- `delay_rate_acceleration` = (lag1 - lag2) - (lag2 - lag3) = lag1 - 2*lag2 + lag3
- This is the second derivative of delay rate
- Captures whether delays are accelerating (+) or decelerating (-)

**Note:** Requires lag3, so will lose one additional month of data compared to baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from calendar import monthrange
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

import holidays

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Generate Holiday Features

In [ ]:
years = list(range(2010, 2026))
STATES = ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

state_holidays = {state: holidays.Australia(years=years, prov=state) for state in STATES}
national_holidays = holidays.Australia(years=years)

def count_holidays_in_month(holiday_dict, year, month):
    count = 0
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        if date(year, month, day) in holiday_dict:
            count += 1
    return count

def has_major_holiday_in_month(holiday_dict, year, month):
    major_keywords = ['christmas', 'easter', 'anzac', 'good friday', 'boxing']
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        d = date(year, month, day)
        if d in holiday_dict:
            name = holiday_dict[d].lower()
            if any(kw in name for kw in major_keywords):
                return 1
    return 0

def get_school_holiday_periods(year):
    return [
        (date(year-1, 12, 18), date(year, 1, 28)),
        (date(year, 4, 8), date(year, 4, 23)),
        (date(year, 6, 27), date(year, 7, 14)),
        (date(year, 9, 23), date(year, 10, 7)),
        (date(year, 12, 18), date(year, 12, 31)),
    ]

def count_school_holiday_days_in_month(year, month):
    periods = get_school_holiday_periods(year)
    if month == 1:
        periods.extend(get_school_holiday_periods(year))
    _, days_in_month = monthrange(year, month)
    holiday_days = 0
    for day in range(1, days_in_month + 1):
        current_date = date(year, month, day)
        for start, end in periods:
            if start <= current_date <= end:
                holiday_days += 1
                break
    return holiday_days

print("Holiday functions defined.")

In [ ]:
year_months = df[['year', 'month_num']].drop_duplicates().sort_values(['year', 'month_num'])
print(f"Generating holiday features for {len(year_months)} unique months...")

holiday_features = []
for _, row in year_months.iterrows():
    year, month = int(row['year']), int(row['month_num'])
    _, days_in_month = monthrange(year, month)
    
    feature_row = {'year': year, 'month_num': month}
    
    # Total holidays (excluding national as redundant)
    all_holiday_dates = set()
    for state in STATES:
        for day in range(1, days_in_month + 1):
            d = date(year, month, day)
            if d in state_holidays[state]:
                all_holiday_dates.add(d)
    feature_row['n_public_holidays_total'] = len(all_holiday_dates)
    
    # Major holiday flag
    has_major = 0
    for state in STATES:
        if has_major_holiday_in_month(state_holidays[state], year, month):
            has_major = 1
            break
    feature_row['has_major_holiday'] = has_major
    
    # School holidays
    school_days = count_school_holiday_days_in_month(year, month)
    feature_row['school_holiday_days'] = school_days
    feature_row['pct_school_holiday'] = round(school_days / days_in_month, 4)
    
    holiday_features.append(feature_row)

df_holidays = pd.DataFrame(holiday_features)
df = df.merge(df_holidays, on=['year', 'month_num'], how='left')
print(f"Shape after adding holidays: {df.shape}")

## 3. Filter Low-Volume Airline-Routes

In [ ]:
volume_threshold = 50
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print(f"Volume threshold: {volume_threshold} flights/month")
print(f"Records before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")

## 4. Create Lag and Acceleration Features

In [ ]:
# Create lag features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_lag3'] = df_filtered.groupby('airline_route')['delay_rate'].shift(3)

# Create gradient (first derivative)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Create acceleration (second derivative)
# acceleration = gradient_t - gradient_{t-1} = (lag1 - lag2) - (lag2 - lag3) = lag1 - 2*lag2 + lag3
df_filtered['delay_rate_acceleration'] = df_filtered['delay_rate_lag1'] - 2*df_filtered['delay_rate_lag2'] + df_filtered['delay_rate_lag3']

print("Acceleration feature statistics:")
print(f"  Min:  {df_filtered['delay_rate_acceleration'].min():.4f}")
print(f"  Max:  {df_filtered['delay_rate_acceleration'].max():.4f}")
print(f"  Mean: {df_filtered['delay_rate_acceleration'].mean():.4f}")
print(f"  Std:  {df_filtered['delay_rate_acceleration'].std():.4f}")

## 5. Explore Acceleration Feature

In [ ]:
# Correlations
print("Feature Correlations with delay_rate:")
print("=" * 50)
print(f"  delay_rate_lag1:         {df_filtered['delay_rate_lag1'].corr(df_filtered['delay_rate']):.4f}")
print(f"  delay_rate_gradient:     {df_filtered['delay_rate_gradient'].corr(df_filtered['delay_rate']):.4f}")
print(f"  delay_rate_acceleration: {df_filtered['delay_rate_acceleration'].corr(df_filtered['delay_rate']):.4f}")

In [ ]:
# By route
print("\nAcceleration Correlation by Route:")
print("=" * 60)
print(f"{'Route':<20} {'lag1':>10} {'gradient':>12} {'acceleration':>14}")
print("-" * 60)

for route in sorted(df_filtered['route'].unique()):
    route_data = df_filtered[df_filtered['route'] == route]
    corr_lag1 = route_data['delay_rate_lag1'].corr(route_data['delay_rate'])
    corr_grad = route_data['delay_rate_gradient'].corr(route_data['delay_rate'])
    corr_accel = route_data['delay_rate_acceleration'].corr(route_data['delay_rate'])
    print(f"{route:<20} {corr_lag1:>10.4f} {corr_grad:>12.4f} {corr_accel:>14.4f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Distribution of acceleration
ax = axes[0]
ax.hist(df_filtered['delay_rate_acceleration'].dropna(), bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', label='Zero (no acceleration)')
ax.set_xlabel('Delay Rate Acceleration')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Acceleration\n(Second Derivative)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Scatter of acceleration vs delay
ax = axes[1]
ax.scatter(df_filtered['delay_rate_acceleration'], df_filtered['delay_rate'], alpha=0.3, s=10)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Delay Rate Acceleration')
ax.set_ylabel('Delay Rate')
ax.set_title('Delay Rate vs Acceleration')
ax.grid(True, alpha=0.3)

# Plot 3: Compare lag1 vs gradient vs acceleration correlations by route
ax = axes[2]
routes = sorted(df_filtered['route'].unique())
corrs_lag1 = [df_filtered[df_filtered['route'] == r]['delay_rate_lag1'].corr(df_filtered[df_filtered['route'] == r]['delay_rate']) for r in routes]
corrs_grad = [df_filtered[df_filtered['route'] == r]['delay_rate_gradient'].corr(df_filtered[df_filtered['route'] == r]['delay_rate']) for r in routes]
corrs_accel = [df_filtered[df_filtered['route'] == r]['delay_rate_acceleration'].corr(df_filtered[df_filtered['route'] == r]['delay_rate']) for r in routes]

x = np.arange(len(routes))
width = 0.25
ax.bar(x - width, corrs_lag1, width, label='lag1', alpha=0.8)
ax.bar(x, corrs_grad, width, label='gradient', alpha=0.8)
ax.bar(x + width, corrs_accel, width, label='acceleration', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '→') for r in routes], rotation=45, ha='right')
ax.set_ylabel('Correlation with delay_rate')
ax.set_title('Feature Correlations by Route')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Cyclical month
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

# Drop NaN - note: acceleration requires lag3, so we lose more data
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_lag3', 
                                       'delay_rate_gradient', 'delay_rate_acceleration']).copy()
print(f"Rows after dropping NaN (requires lag3): {len(df_clean)}")

## 7. Train/Validation/Test Split

In [ ]:
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

# Baseline = recommended features from 7a-7c
baseline_features = base_features + weather_features + holiday_features

# Variants to test
accel_variants = {
    'baseline': baseline_features,
    '+acceleration': baseline_features + ['delay_rate_acceleration'],
}

print("Feature variants:")
for name, features in accel_variants.items():
    print(f"  {name}: {len(features)} features")

In [ ]:
# Target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 8. Regression Models

In [ ]:
reg_results = []

for variant_name, features in accel_variants.items():
    print(f"\n{'='*60}")
    print(f"Regression with: {variant_name}")
    print(f"{'='*60}")
    
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    reg_results.append({'Variant': variant_name, 'Model': 'Ridge', 'Test_R2': ridge_r2, 'Test_RMSE': ridge_rmse})
    
    # Random Forest
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    reg_results.append({'Variant': variant_name, 'Model': 'Random Forest', 'Test_R2': rf_r2, 'Test_RMSE': rf_rmse})

reg_df = pd.DataFrame(reg_results)

## 9. Classification Models

In [ ]:
clf_results = []

for variant_name, features in accel_variants.items():
    print(f"\n{'='*60}")
    print(f"Classification with: {variant_name}")
    print(f"{'='*60}")
    
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    clf_results.append({'Variant': variant_name, 'Model': 'Logistic', 'Test_F1': logreg_f1, 'Test_AUC': logreg_auc})
    
    # Random Forest
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    clf_results.append({'Variant': variant_name, 'Model': 'Random Forest', 'Test_F1': rf_clf_f1, 'Test_AUC': rf_clf_auc})
    
    # XGBoost
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        clf_results.append({'Variant': variant_name, 'Model': 'XGBoost', 'Test_F1': xgb_f1, 'Test_AUC': xgb_auc})

clf_df = pd.DataFrame(clf_results)

## 10. Results Comparison

In [ ]:
# Regression comparison
print("=" * 70)
print("REGRESSION: Baseline vs +Acceleration")
print("=" * 70)

reg_pivot = reg_df.pivot(index='Model', columns='Variant', values='Test_R2')
reg_pivot = reg_pivot[['baseline', '+acceleration']]

print(f"\n{'Model':<15} {'baseline':>12} {'+accel':>12} {'Δ R²':>10}")
print("-" * 50)
for model in reg_pivot.index:
    baseline = reg_pivot.loc[model, 'baseline']
    accel = reg_pivot.loc[model, '+acceleration']
    diff = accel - baseline
    sign = '+' if diff > 0 else ''
    print(f"{model:<15} {baseline:>12.4f} {accel:>12.4f} {sign}{diff:>9.4f}")

In [ ]:
# Classification comparison
print("=" * 70)
print("CLASSIFICATION: Baseline vs +Acceleration")
print("=" * 70)

clf_pivot = clf_df.pivot(index='Model', columns='Variant', values='Test_F1')
clf_pivot = clf_pivot[['baseline', '+acceleration']]

print(f"\n{'Model':<15} {'baseline':>12} {'+accel':>12} {'Δ F1':>10}")
print("-" * 50)
for model in clf_pivot.index:
    baseline = clf_pivot.loc[model, 'baseline']
    accel = clf_pivot.loc[model, '+acceleration']
    diff = accel - baseline
    sign = '+' if diff > 0 else ''
    print(f"{model:<15} {baseline:>12.4f} {accel:>12.4f} {sign}{diff:>9.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Regression
ax = axes[0]
x_reg = np.arange(len(reg_pivot.index))
width = 0.35
colors = ['steelblue', '#27ae60']

ax.bar(x_reg - width/2, reg_pivot['baseline'], width, label='baseline', color=colors[0], alpha=0.8)
ax.bar(x_reg + width/2, reg_pivot['+acceleration'], width, label='+acceleration', color=colors[1], alpha=0.8)

ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x_reg)
ax.set_xticklabels(reg_pivot.index)
ax.legend()
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3, axis='y')

# Classification
ax = axes[1]
x_clf = np.arange(len(clf_pivot.index))

ax.bar(x_clf - width/2, clf_pivot['baseline'], width, label='baseline', color=colors[0], alpha=0.8)
ax.bar(x_clf + width/2, clf_pivot['+acceleration'], width, label='+acceleration', color=colors[1], alpha=0.8)

ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x_clf)
ax.set_xticklabels(clf_pivot.index)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 11. Feature Importance

In [ ]:
# Train RF with acceleration to check feature importance
features_with_accel = baseline_features + ['delay_rate_acceleration']
X_train_reg = df_clean.loc[train_mask, features_with_accel].values

rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(X_train_reg, y_train_reg)

importance_df = pd.DataFrame({
    'Feature': features_with_accel,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF Regression with acceleration):")
print("-" * 60)
for _, row in importance_df.head(15).iterrows():
    marker = " <- ACCEL" if 'acceleration' in row['Feature'] else ""
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}{marker}")

# Find acceleration rank
accel_rank = list(importance_df['Feature']).index('delay_rate_acceleration') + 1
accel_importance = importance_df[importance_df['Feature'] == 'delay_rate_acceleration']['Importance'].values[0]
print(f"\ndelay_rate_acceleration: rank {accel_rank}, importance {accel_importance:.4f}")

## 12. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Impact of Delay Rate Acceleration on Multi-Route Model")
print("="*80)

# Summary metrics table - Regression
print("\nREGRESSION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<15} {'Baseline':>12} {'+Accel':>12} {'Δ R²':>10} {'Status':>12}")
print("-" * 70)

for model in reg_pivot.index:
    baseline = reg_pivot.loc[model, 'baseline']
    accel = reg_pivot.loc[model, '+acceleration']
    diff = accel - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {accel:>12.4f} {sign}{diff:>9.4f} {status:>12}")

# Summary metrics table - Classification
print("\nCLASSIFICATION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<15} {'Baseline':>12} {'+Accel':>12} {'Δ F1':>10} {'Status':>12}")
print("-" * 70)

for model in clf_pivot.index:
    baseline = clf_pivot.loc[model, 'baseline']
    accel = clf_pivot.loc[model, '+acceleration']
    diff = accel - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {accel:>12.4f} {sign}{diff:>9.4f} {status:>12}")



### Observations:

- Requires lag3, which means one additional month lost per airline-route
   - Baseline only uses up to lag2
- No improvement to the regression models
- Very minor improvement to the classification models
- This indicates the first derivative (gradient) may already capture sufficient trend info

## 13. Next Step

Consolidate all of the features explored in notebooks 6e to 7d (only include the ones with notable positive impact). Will start with training data preparation with the new features included.